In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read Bronze table
bronze_df = spark.table(
    "workspace.movies_dev.bronze_movies"
)

# Define duplicate identification
window_spec = (
    Window
    .partitionBy("title", "release_year", "industry")
    .orderBy(F.col("ingestion_timestamp").asc_nulls_last())
)

# Identify duplicates
df = bronze_df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
)

# Define validation conditions
df = df.withColumn(
    "quarantine_reason",
    F.concat_ws(
        " | ",
        F.when(
            F.col("title").isNull() | (F.trim("title") == ""),
            "Missing movie title"
        ),
        F.when(
            F.col("release_year").isNull(),
            "Missing release year"
        ),
        F.when(
            ~F.col("release_year").between(1888, 2026),
            "Invalid release year"
        ),
        F.when(
            F.col("imdb_rating").isNotNull() &
            ~F.expr("try_cast(imdb_rating as double)").between(0, 10),
            "Invalid IMDb rating"
        ),
        F.when(
            F.col("budget").isNotNull() &
            (F.col("budget") < 0),
            "Negative budget"
        ),
        F.when(
            F.col("revenue").isNotNull() &
            (F.col("revenue") < 0),
            "Negative revenue"
        ),
        F.when(
            F.col("row_num") > 1,
            "Duplicate movie record"
        )
    )
)

# Separate valid and invalid records
valid_df = df.filter(
    F.col("quarantine_reason") == ""
).drop("row_num", "quarantine_reason")

quarantine_df = (
    df.filter(F.col("quarantine_reason") != "")
    .drop("row_num")
    .withColumn("quarantined_at", F.current_timestamp())
)

# Standardize valid records
silver_df = (
    valid_df
    .withColumn("title", F.trim("title"))
    .withColumn("industry", F.upper(F.trim("industry")))
    .withColumn("studio", F.trim("studio"))
    .withColumn("language", F.lower(F.trim("language")))
    .withColumn("currency", F.upper(F.trim("currency")))
)

# Write Silver table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.movies_dev.silver_movies")
)

# Write Quarantine table
(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.movies_dev.quarantine_movies")
)

print("Silver transformation completed")
print("Valid records:", silver_df.count())
print("Quarantined records:", quarantine_df.count())

display(quarantine_df)


In [0]:
%sql
SELECT COUNT(*)
FROM workspace.movies_dev.silver_movies;

In [0]:
%sql
SELECT *
FROM workspace.movies_dev.quarantine_movies;

In [0]:
%sql
SELECT
    quarantine_reason,
    COUNT(*) AS rejected_records
FROM workspace.movies_dev.quarantine_movies
GROUP BY quarantine_reason;